In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("No GPU detected, using CPU")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU device: NVIDIA A100-SXM4-80GB
Number of GPUs: 1


In [2]:
!pip install roboflow
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.7/91.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 142.3 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 64.8 MB/s eta 0:00:00


In [3]:
from roboflow import Roboflow
from google.colab import userdata
import os

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("renaire-odarve")
project = workspace.project("erotica-detection")
version = project.version(4) # version of dataset in roboflow
dataset = version.download("voc")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to erotica-detection-4 in voc:: 100%|██████████| 31621/31621 [00:03<00:00, 8295.60it/s]


In [4]:
import os
import torch
from PIL import Image
import xml.etree.ElementTree as ET

CLASSES = ["__background__", "nsfw", "safe", "gore"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)  # = 4

class RoboflowVOCDataset(torch.utils.data.Dataset):
    def __init__(self, root, split="train", transforms=None):
        self.root = root
        self.split = split
        self.transforms = transforms
        self.split_dir = os.path.join(root, split)

        self.images = sorted(
            f for f in os.listdir(self.split_dir) if f.endswith(".jpg")
        )

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.split_dir, img_name)
        xml_path = img_path.replace(".jpg", ".xml")

        image = Image.open(img_path).convert("RGB")

        boxes = []
        labels = []

        if os.path.exists(xml_path):
            tree = ET.parse(xml_path)
            root = tree.getroot()

            for obj in root.findall("object"):
                class_name = obj.find("name").text.strip().lower()

                if class_name not in CLASS_TO_IDX:
                    continue

                label = CLASS_TO_IDX[class_name]

                bndbox = obj.find("bndbox")
                xmin = float(bndbox.find("xmin").text)
                ymin = float(bndbox.find("ymin").text)
                xmax = float(bndbox.find("xmax").text)
                ymax = float(bndbox.find("ymax").text)

                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(label)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        if self.transforms:
            image, target = self.transforms(image, target)

        return image, target

    def __len__(self):
        return len(self.images)

In [ ]:
import torchvision.transforms.functional as F
import torch
import random
from PIL import Image

class SSDTransform:
    # NOTE: We deliberately do NOT normalize here. torchvision's SSD has a
    # built-in GeneralizedRCNNTransform that normalizes every input with
    # ImageNet mean/std (image_mean=[0.485,0.456,0.406], image_std=[0.229,0.224,0.225]).
    # Normalizing here too would normalize each image twice during training,
    # while the exported model only normalizes once -> a train/inference
    # mismatch. We output plain [0,1] tensors and let the model normalize.
    def __init__(self, size=(640, 640), split='train'):
        self.size = size
        self.split = split

    def __call__(self, image, target):
        # image: PIL Image
        # target: dict with 'boxes', 'labels'

        # === Augmentations (Train Only) ===
        if self.split == 'train':
            # 1. Color Jitter
            if random.random() < 0.5:
                image = F.adjust_brightness(image, brightness_factor=random.uniform(0.6, 1.4))
                image = F.adjust_contrast(image, contrast_factor=random.uniform(0.6, 1.4))
                image = F.adjust_saturation(image, saturation_factor=random.uniform(0.6, 1.4))
                image = F.adjust_hue(image, hue_factor=random.uniform(-0.1, 0.1))

            # 2. Random Horizontal Flip
            if random.random() < 0.5:
                image = F.hflip(image)
                w, h = image.size
                boxes = target['boxes']
                if boxes.numel() > 0:
                    # Flip x coordinates: x1_new = w - x2_old
                    boxes[:, [0, 2]] = w - boxes[:, [2, 0]]
                    target['boxes'] = boxes

            # 3. Scale Jitter / Zoom Out
            # Expands the image canvas (making the objects smaller relative to the new frame)
            if random.random() < 0.5:
                scale = random.uniform(1.0, 1.5)
                w, h = image.size
                new_w = int(w * scale)
                new_h = int(h * scale)

                # Create larger gray canvas
                new_image = Image.new("RGB", (new_w, new_h), (123, 117, 104))
                left = random.randint(0, new_w - w)
                top = random.randint(0, new_h - h)
                new_image.paste(image, (left, top))

                image = new_image

                # Shift boxes
                boxes = target['boxes']
                if boxes.numel() > 0:
                    boxes[:, [0, 2]] += left
                    boxes[:, [1, 3]] += top
                    target['boxes'] = boxes

        # === Resizing (All Splits) ===
        # Resize to a fixed square; normalization is handled by the model's
        # internal transform, so we only convert to a [0,1] tensor here.
        w, h = image.size
        image = F.resize(image, self.size)
        image = F.to_tensor(image)

        # Resize bounding boxes
        boxes = target['boxes']
        if boxes.numel() > 0:
            scale_w = self.size[1] / w
            scale_h = self.size[0] / h

            boxes[:, 0] *= scale_w
            boxes[:, 2] *= scale_w
            boxes[:, 1] *= scale_h
            boxes[:, 3] *= scale_h

            # Clamp boxes to ensure they stay within image bounds
            boxes[:, 0].clamp_(min=0, max=self.size[1])
            boxes[:, 1].clamp_(min=0, max=self.size[0])
            boxes[:, 2].clamp_(min=0, max=self.size[1])
            boxes[:, 3].clamp_(min=0, max=self.size[0])

            target['boxes'] = boxes

        return image, target

# Create separate transforms
train_transform = SSDTransform(split='train')
val_transform = SSDTransform(split='valid')

In [6]:
# Use the location from the downloaded dataset object if available, otherwise default to the expected Colab path
try:
    dataset_root = dataset.location
except NameError:
    dataset_root = "/content/erotica-detection-4"


train_dataset = RoboflowVOCDataset(
    root=dataset_root,
    split="train",
    transforms=train_transform, # Use train_transform
)

val_dataset = RoboflowVOCDataset(
    root=dataset_root,
    split="valid",
    transforms=val_transform, # Use val_transform
)

In [7]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    # Keep every image, including negatives with no boxes. torchvision's SSD
    # assigns all anchors to background for empty targets, and these "safe"
    # images are the hard negatives that suppress false positives.
    return tuple(zip(*batch))


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True,
    num_workers=8
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=8
)

In [9]:
import torch
import torch.nn as nn
from functools import partial
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection import SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models.detection.ssdlite import SSDLiteHead

# 1. Load the model with COCO weights (implicitly sets num_classes=91)
# Set trainable_backbone_layers=6 to unfreeze the entire backbone
model = ssdlite320_mobilenet_v3_large(
    weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1,
    trainable_backbone_layers=6
)

# Explicit check to unfreeze just in case
for param in model.backbone.parameters():
    param.requires_grad = True

# Override internal transform to allow 640x640 input
model.transform.min_size = (640,)
model.transform.max_size = 640

# 2. Get the input channels and anchors from the existing model
# Helper function to recursively find the first Conv2d layer's input channels
def get_in_channels(module):
    for layer in module.modules():
        if isinstance(layer, nn.Conv2d):
            return layer.in_channels
    raise ValueError(f"Could not find Conv2d in {module}")

# We extract these to ensure the new head matches the backbone's output
in_channels = [get_in_channels(layer) for layer in model.head.classification_head.module_list]
num_anchors = model.anchor_generator.num_anchors_per_location()

# 3. Create a new head for our specific number of classes (4)
new_head = SSDLiteHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=NUM_CLASSES,  # 4
    norm_layer=partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
)

# 4. Replace the old head with the new one
model.head = new_head

print("Model head successfully replaced for 4 classes (Input size: 640x640). Backbone is unfrozen.")

Downloading: "https://download.pytorch.org/models/ssdlite320_mobilenet_v3_large_coco-a79551df.pth" to /root/.cache/torch/hub/checkpoints/ssdlite320_mobilenet_v3_large_coco-a79551df.pth


100%|██████████| 13.4M/13.4M [00:00<00:00, 221MB/s]

Model head successfully replaced for 4 classes (Input size: 640x640). Backbone is unfrozen.


In [10]:
print(model.head.classification_head)


SSDLiteClassificationHead(
  (module_list): ModuleList(
    (0): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(672, 672, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=672, bias=False)
        (1): BatchNorm2d(672, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): Conv2d(672, 24, kernel_size=(1, 1), stride=(1, 1))
    )
    (1): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(480, 480, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=480, bias=False)
        (1): BatchNorm2d(480, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
      )
      (1): Conv2d(480, 24, kernel_size=(1, 1), stride=(1, 1))
    )
    (2): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=512, bias=False)
        (1): BatchNorm2d(512, eps=0.001, momentum=0.03, aff

In [12]:
import torch.nn as nn

def replace_bn_with_gn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            groups = min(32, child.num_features)
            while child.num_features % groups != 0:
                groups -= 1
            setattr(module, name, nn.GroupNorm(groups, child.num_features))
        else:
            replace_bn_with_gn(child)


replace_bn_with_gn(model)


In [13]:
images, targets = next(iter(train_loader))

print(images[0].shape)            # [3, 320, 320]
print(targets[0]["boxes"].shape)  # [N, 4] or [0, 4]
print(targets[0]["labels"])       # tensor([1, 2]) or empty


torch.Size([3, 640, 640])
torch.Size([2, 4])
tensor([1, 1])


In [14]:
import torch.optim as optim

optimizer = optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.9,
    weight_decay=0.0005
)


In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model.to(device)

Using device: cuda


SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): GroupNorm(16, 16, eps=1e-05, affine=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): GroupNorm(16, 16, eps=1e-05, affine=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): GroupNorm(16, 16, eps=1e-05, affine=True)
            )
          )
        )
        (2): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 64, k

In [17]:
import time
import torch
import os
import csv
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.optim.lr_scheduler import CosineAnnealingLR

# Create results folder
os.makedirs("results", exist_ok=True)
csv_path = os.path.join("results", "metrics.csv")

# Initialize CSV with expanded headers
headers = [
    "Epoch", "Train Loss", "LR",
    "Train Speed (img/s)",
    "Val mAP@50", "Val mAP@50-95", "Val mAP@75",
    "Val Latency (ms)", "Val Speed (img/s)"
]
with open(csv_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(headers)

num_epochs = 150
# Set max_detection_thresholds to [200] as requested
# Fix: max_detection_thresholds expects a list of 3 integers for mAP, mAP@50, and mAP@75.
# Common values for COCO evaluation are [1, 10, 100].
map_metric = MeanAveragePrecision(max_detection_thresholds=[1, 10, 100]).cpu()

# Initialize scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    num_train_images = 0
    start_time = time.time()

    for batch in train_loader:
        if batch is None:
            continue

        images, targets = batch

        # Count images for accurate speed calc
        batch_size = len(images)
        num_train_images += batch_size

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        cls_loss = loss_dict["classification"]
        box_loss = loss_dict["bbox_regression"]

        loss = cls_loss * 1.5 + box_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

    # Step the scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    epoch_loss /= max(1, num_batches)

    train_time = time.time() - start_time
    train_speed = num_train_images / train_time if train_time > 0 else 0

    print(
        f"\nEpoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f} | "
        f"LR: {current_lr:.6f} | "
        f"Speed: {train_speed:.2f} imgs/sec"
    )

    # ==========================
    # Validation (mAP + Latency)
    # ==========================
    map_50, map_50_95, map_75 = "", "", ""
    val_latency, val_speed = "", ""

    if (epoch + 1) % 5 == 0:
        model.eval()
        map_metric.reset()

        val_start = time.time()
        num_val_images = 0

        with torch.no_grad():
            for batch in val_loader:
                if batch is None:
                    continue

                images, targets = batch
                num_val_images += len(images)

                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                outputs = model(images)

                outputs = [{k: v.cpu() for k, v in o.items()} for o in outputs]
                targets = [{k: v.cpu() for k, v in t.items()} for t in targets]

                map_metric.update(outputs, targets)

        val_duration = time.time() - val_start

        if num_val_images > 0:
            val_speed = num_val_images / val_duration
            val_latency = (val_duration * 1000) / num_val_images # ms per image

        metrics = map_metric.compute()
        map_50 = metrics['map_50'].item()
        map_50_95 = metrics['map'].item()
        map_75 = metrics['map_75'].item()

        print(f"Validation -> mAP@50: {map_50:.4f} | mAP@50-95: {map_50_95:.4f} | Latency: {val_latency:.2f} ms")

    # Save to CSV
    with open(csv_path, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            epoch+1, epoch_loss, current_lr,
            f"{train_speed:.2f}",
            map_50, map_50_95, map_75,
            f"{val_latency:.2f}" if isinstance(val_latency, float) else "",
            f"{val_speed:.2f}" if isinstance(val_speed, float) else ""
        ])

# ==========================
# Test Metrics (Optional)
# ==========================
print("\nRunning Test Evaluation...")
try:
    # Try to initialize test dataset
    test_dataset = RoboflowVOCDataset(
        root=dataset_root,
        split="test",
        transforms=val_transform # Use validation transform for testing
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8
    )

    model.eval()
    map_metric.reset()
    test_start = time.time()
    num_test_imgs = 0

    with torch.no_grad():
        for batch in test_loader:
            if batch is None: continue
            images, targets = batch
            num_test_imgs += len(images)
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            outputs = model(images)
            outputs = [{k: v.cpu() for k, v in o.items()} for o in outputs]
            targets = [{k: v.cpu() for k, v in t.items()} for t in targets]
            map_metric.update(outputs, targets)

    test_duration = time.time() - test_start
    test_metrics = map_metric.compute()

    print(f"\n=== TEST RESULTS ===")
    print(f"mAP@50: {test_metrics['map_50'].item():.4f}")
    print(f"mAP@50-95: {test_metrics['map'].item():.4f}")
    print(f"mAP@75: {test_metrics['map_75'].item():.4f}")
    print(f"Inference Latency: {(test_duration * 1000) / num_test_imgs:.2f} ms/img")
    print(f"Test Speed: {num_test_imgs / test_duration:.2f} imgs/sec")

    # Save test metrics separately
    with open(os.path.join("results", "test_metrics.csv"), "w") as f:
        f.write("metric,value\n")
        f.write(f"map_50,{test_metrics['map_50'].item()}\n")
        f.write(f"map_50_95,{test_metrics['map'].item()}\n")
        f.write(f"latency_ms,{(test_duration * 1000) / num_test_imgs}\n")

except Exception as e:
    print(f"\nSkipping Test Evaluation: {e}")


Epoch [1/150] Loss: 7.5192 | LR: 0.002000 | Speed: 125.91 imgs/sec

Epoch [2/150] Loss: 6.4274 | LR: 0.001999 | Speed: 126.23 imgs/sec

Epoch [3/150] Loss: 6.0395 | LR: 0.001998 | Speed: 126.90 imgs/sec

Epoch [4/150] Loss: 5.8030 | LR: 0.001996 | Speed: 127.08 imgs/sec

Epoch [5/150] Loss: 5.6475 | LR: 0.001995 | Speed: 126.36 imgs/sec


Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.


Validation -> mAP@50: 0.3546 | mAP@50-95: 0.1244 | Latency: 7.02 ms

Epoch [6/150] Loss: 5.5238 | LR: 0.001992 | Speed: 125.09 imgs/sec

Epoch [7/150] Loss: 5.4125 | LR: 0.001989 | Speed: 124.87 imgs/sec

Epoch [8/150] Loss: 5.3360 | LR: 0.001986 | Speed: 124.64 imgs/sec

Epoch [9/150] Loss: 5.2340 | LR: 0.001982 | Speed: 125.37 imgs/sec

Epoch [10/150] Loss: 5.1652 | LR: 0.001978 | Speed: 123.73 imgs/sec
Validation -> mAP@50: 0.4610 | mAP@50-95: 0.1821 | Latency: 7.15 ms

Epoch [11/150] Loss: 5.1320 | LR: 0.001974 | Speed: 123.82 imgs/sec

Epoch [12/150] Loss: 5.0708 | LR: 0.001969 | Speed: 124.34 imgs/sec

Epoch [13/150] Loss: 5.0121 | LR: 0.001963 | Speed: 124.33 imgs/sec

Epoch [14/150] Loss: 4.9704 | LR: 0.001957 | Speed: 123.25 imgs/sec

Epoch [15/150] Loss: 4.9315 | LR: 0.001951 | Speed: 123.84 imgs/sec
Validation -> mAP@50: 0.4973 | mAP@50-95: 0.1964 | Latency: 7.14 ms

Epoch [16/150] Loss: 4.8905 | LR: 0.001944 | Speed: 124.47 imgs/sec

Epoch [17/150] Loss: 4.8516 | LR: 0.0019

Encountered more than 100 detections in a single image. This means that certain detections with the lowest scores will be ignored, that may have an undesirable impact on performance. Please consider adjusting the `max_detection_threshold` to suit your use case. To disable this warning, set attribute class `warn_on_many_detections=False`, after initializing the metric.



=== TEST RESULTS ===
mAP@50: 0.5729
mAP@50-95: 0.2465
mAP@75: 0.1726
Inference Latency: 7.32 ms/img
Test Speed: 136.69 imgs/sec


In [18]:
# Zip the results folder
!zip -r ssdmobilenet_results.zip /content/results/

  adding: content/results/ (stored 0%)
  adding: content/results/metrics.csv (deflated 54%)
  adding: content/results/test_metrics.csv (deflated 14%)


In [19]:
from google.colab import files

files.download('ssdmobilenet_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import os
import torch
import json
import textwrap

# 1. Create conversion folder
output_dir = "conversion"
os.makedirs(output_dir, exist_ok=True)

# 2. Save Trained Weights & Checkpoint
# Save weights only
torch.save(model.state_dict(), os.path.join(output_dir, "ssd_mbv3_large.pth"))

# Save full checkpoint
torch.save({
    "epoch": num_epochs,
    "model": model.state_dict(),
    "optimizer": optimizer.state_dict(),
    "classes": CLASSES,
}, os.path.join(output_dir, "ssd_mbv3_large_full.pth"))

# 3. Save Class Labels
with open(os.path.join(output_dir, "labels.json"), "w") as f:
    json.dump(CLASSES, f, indent=4)

# 4. Save Model Architecture Code
# We will write a python script that can reconstruct the model
arch_code = """
import torch
import torch.nn as nn
from functools import partial
from torchvision.models.detection import ssdlite320_mobilenet_v3_large
from torchvision.models.detection.ssdlite import SSDLiteHead

def replace_bn_with_gn(module):
    for name, child in module.named_children():
        if isinstance(child, nn.BatchNorm2d):
            groups = min(32, child.num_features)
            while child.num_features % groups != 0:
                groups -= 1
            setattr(module, name, nn.GroupNorm(groups, child.num_features))
        else:
            replace_bn_with_gn(child)

def get_in_channels(module):
    for layer in module.modules():
        if isinstance(layer, nn.Conv2d):
            return layer.in_channels
    raise ValueError(f"Could not find Conv2d in {module}")

def create_model(num_classes=4, input_size=640):
    # 1. Load the base model structure (weights don't matter if loading state_dict later)
    model = ssdlite320_mobilenet_v3_large(weights=None, trainable_backbone_layers=6)

    # 2. Override internal transform size
    model.transform.min_size = (input_size,)
    model.transform.max_size = input_size

    # 3. Reconstruct the head
    in_channels = [get_in_channels(layer) for layer in model.head.classification_head.module_list]
    num_anchors = model.anchor_generator.num_anchors_per_location()

    new_head = SSDLiteHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
        norm_layer=partial(nn.BatchNorm2d, eps=0.001, momentum=0.03)
    )
    model.head = new_head

    # 4. Replace BN with GN
    replace_bn_with_gn(model)

    return model

if __name__ == "__main__":
    model = create_model()
    print("Model created successfully")
"""

with open(os.path.join(output_dir, "model_arch.py"), "w") as f:
    f.write(arch_code)

print(f"All artifacts saved to {output_dir}/:")
print(f"1. ssd_mbv3_large.pth (Weights)")
print(f"2. ssd_mbv3_large_full.pth (Full Checkpoint)")
print(f"3. labels.json (Classes)")
print(f"4. model_arch.py (Architecture Code)")

All artifacts saved to conversion/:
1. ssd_mbv3_large.pth (Weights)
2. ssd_mbv3_large_full.pth (Full Checkpoint)
3. labels.json (Classes)
4. model_arch.py (Architecture Code)


In [21]:
import json
import os

# Define Training Configuration
train_config = {
    "model_architecture": "ssdlite320_mobilenet_v3_large",
    "dataset": "Roboflow VOC",
    "num_classes": NUM_CLASSES,
    "class_names": CLASSES,
    "input_size": 640,
    "batch_size": 32,
    "epochs": num_epochs,
    "optimizer": {
        "type": "SGD",
        "lr": 0.002,
        "momentum": 0.9,
        "weight_decay": 0.0005
    },
    "scheduler": "CosineAnnealingLR",
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

# Save to the conversion directory
config_path = os.path.join("conversion", "training_config.json")
with open(config_path, "w") as f:
    json.dump(train_config, f, indent=4)

print(f"Training configuration saved to {config_path}")

Training configuration saved to conversion/training_config.json


In [22]:
# Zip the results folder
!zip -r ssdmobilenet_conversion.zip /content/conversion/

  adding: content/conversion/ (stored 0%)
  adding: content/conversion/training_config.json (deflated 44%)
  adding: content/conversion/model_arch.py (deflated 57%)
  adding: content/conversion/labels.json (deflated 27%)
  adding: content/conversion/ssd_mbv3_large_full.pth (deflated 7%)
  adding: content/conversion/ssd_mbv3_large.pth (deflated 8%)


In [23]:
from google.colab import files

files.download('ssdmobilenet_conversion.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
!pip install onnx onnxruntime onnxscript


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 17.6 MB/s eta 0:00:00


In [ ]:
import torch

# ONNX export must run with the model AND the dummy input on the SAME device.
# The previous version failed because the model was on CUDA while `dummy` was on
# CPU, triggering:
#   RuntimeError: Unhandled FakeTensor Device Propagation for
#   aten.native_group_norm.default, found two different devices cpu, cuda:0
#
# We export on CPU and use the legacy TorchScript exporter (dynamo=False).
# The new dynamo exporter cannot trace torchvision's SSD post-processing
# (it errors with "always_classified is unsupported").

model.eval()
model.cpu()                            # move the whole model (incl. GroupNorm) to CPU
dummy = torch.randn(1, 3, 640, 640)    # CPU tensor, matching the model

torch.onnx.export(
    model,
    dummy,
    "ssd_mbv3_150e.onnx",
    input_names=["input"],
    output_names=["boxes", "scores", "labels"],
    opset_version=18,
    dynamo=False,
)
print("Exported ssd_mbv3_150e.onnx")

# Move the model back to the training device in case more cells run afterwards.
model.to(device)
